# PySpark practice 2: DataFrame API

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-practice-02-dataframe-api.ipynb)

16 exercises on the DataFrame API: columns, filters, deduplication, sorting, unions, set operations, transform and schemas. Each exercise names the [reference entries](pyspark-reference-02-dataframe-api.ipynb) it practises, shows its inputs and the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 pandas pyarrow

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and the helpers from [PySpark reference 2](pyspark-reference-02-dataframe-api.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        print(type(error).__name__ + ":", str(error).split(".")[0][:100])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Exercises

9 easy, 5 medium, 2 hard. Write your answer in the empty cell under each exercise as a DataFrame called `result`, run it, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Select, compute and rename](#exercise-1-select-compute-and-rename) | Easy | [`select(*cols)` and `selectExpr(*expr)`](pyspark-reference-02-dataframe-api.ipynb#selectcols-and-selectexprexpr) |
| 2 | [Add several derived columns at once](#exercise-2-add-several-derived-columns-at-once) | Easy | [`withColumn(colName, col)` and `withColumns(colsMap)`](pyspark-reference-02-dataframe-api.ipynb#withcolumncolname-col-and-withcolumnscolsmap) |
| 3 | [Normalize messy column names](#exercise-3-normalize-messy-column-names) | Medium | [Renaming: `withColumnRenamed`, `withColumnsRenamed`, `toDF`, `alias`](pyspark-reference-02-dataframe-api.ipynb#renaming-withcolumnrenamed-withcolumnsrenamed-todf-alias) |
| 4 | [Select columns with awkward names](#exercise-4-select-columns-with-awkward-names) | Easy | [Referring to columns](pyspark-reference-02-dataframe-api.ipynb#referring-to-columns) |
| 5 | [Keep rows where a nullable column is not a value](#exercise-5-keep-rows-where-a-nullable-column-is-not-a-value) | Easy | [`where(condition)` / `filter(condition)`](pyspark-reference-02-dataframe-api.ipynb#wherecondition--filtercondition) |
| 6 | [Remove exact duplicate events](#exercise-6-remove-exact-duplicate-events) | Easy | [`distinct()` and `dropDuplicates(subset=None)`](pyspark-reference-02-dataframe-api.ipynb#distinct-and-dropduplicatessubsetnone) |
| 7 | [Unique values in a column](#exercise-7-unique-values-in-a-column) | Easy | [`distinct()` and `dropDuplicates(subset=None)`](pyspark-reference-02-dataframe-api.ipynb#distinct-and-dropduplicatessubsetnone) |
| 8 | [Merge feeds with different column orders](#exercise-8-merge-feeds-with-different-column-orders) | Medium | [`union`, `unionAll` and `unionByName`](pyspark-reference-02-dataframe-api.ipynb#union-unionall-and-unionbyname) |
| 9 | [Compare two snapshots including duplicates](#exercise-9-compare-two-snapshots-including-duplicates) | Medium | [`intersect`, `subtract` and `exceptAll`](pyspark-reference-02-dataframe-api.ipynb#intersect-subtract-and-exceptall) |
| 10 | [Customers present in both lists](#exercise-10-customers-present-in-both-lists) | Easy | [`intersect`, `subtract` and `exceptAll`](pyspark-reference-02-dataframe-api.ipynb#intersect-subtract-and-exceptall) |
| 11 | [Paginate a sorted result](#exercise-11-paginate-a-sorted-result) | Medium | [`limit(n)`, `offset(n)` and `tail(n)`](pyspark-reference-02-dataframe-api.ipynb#limitn-offsetn-and-tailn) |
| 12 | [Sort with nulls last and a tie-breaker](#exercise-12-sort-with-nulls-last-and-a-tie-breaker) | Easy | [`orderBy(*cols, ascending=True)` / `sort(...)`](pyspark-reference-02-dataframe-api.ipynb#orderbycols-ascendingtrue--sort) |
| 13 | [Build a pipeline with transform](#exercise-13-build-a-pipeline-with-transform) | Medium | [`DataFrame.transform(func, *args, **kwargs)`](pyspark-reference-02-dataframe-api.ipynb#dataframetransformfunc-args-kwargs) |
| 14 | [Summary statistics for a column](#exercise-14-summary-statistics-for-a-column) | Easy | [`describe(*cols)` and `summary(*statistics)`](pyspark-reference-02-dataframe-api.ipynb#describecols-and-summarystatistics) |
| 15 | [Create a nested DataFrame from Python data](#exercise-15-create-a-nested-dataframe-from-python-data) | Hard | [Schemas: `StructType`, `StructField` and DDL strings](pyspark-reference-02-dataframe-api.ipynb#schemas-structtype-structfield-and-ddl-strings) |
| 16 | [Detect schema drift between two versions](#exercise-16-detect-schema-drift-between-two-versions) | Hard | [Schemas: `StructType`, `StructField` and DDL strings](pyspark-reference-02-dataframe-api.ipynb#schemas-structtype-structfield-and-ddl-strings) |

## Exercise 1: Select, compute and rename

**Difficulty:** Easy · about 5 min · **Practises:** [`select(*cols)` and `selectExpr(*expr)`](pyspark-reference-02-dataframe-api.ipynb#selectcols-and-selectexprexpr)

*The bread and butter of every transformation step.*

For paid orders, return `order_id`, `amount_eur` (the `amount` column renamed) and `vat` (20% of amount, rounded to 2 decimals), ordered by `order_id`.

In [ ]:
orders.select("order_id", "status", "amount").show()

**Expected output**

| order_id | amount_eur | vat |
| --- | --- | --- |
| 1 | 120.00 | 24.0 |
| 4 | 260.00 | 52.0 |
| 7 | 310.00 | 62.0 |
| 10 | 74.00 | 14.8 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = (
    orders.where(F.col("status") == "paid")
    .select("order_id", F.col("amount").alias("amount_eur"), F.round(F.col("amount") * 0.2, 2).alias("vat"))
    .orderBy("order_id")
)
result.show()

## Exercise 2: Add several derived columns at once

**Difficulty:** Easy · about 5 min · **Practises:** [`withColumn(colName, col)` and `withColumns(colsMap)`](pyspark-reference-02-dataframe-api.ipynb#withcolumncolname-col-and-withcolumnscolsmap)

*Silver-layer steps add date parts and flags in one go.*

Add `order_date` (date of `order_ts`), `order_hour` (hour) and `is_large` (amount >= 200) with one `withColumns` call. Return `order_id`, `order_date`, `order_hour`, `is_large` for orders 1 to 4, ordered by `order_id`.

In [ ]:
orders.select("order_id", "order_ts", "amount").show()

**Expected output**

| order_id | order_date | order_hour | is_large |
| --- | --- | --- | --- |
| 1 | 2026-01-05 | 9 | false |
| 2 | 2026-01-05 | 10 | false |
| 3 | 2026-01-05 | 11 | false |
| 4 | 2026-01-06 | 15 | true |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = (
    orders.withColumns({"order_date": F.to_date("order_ts"), "order_hour": F.hour("order_ts"), "is_large": F.col("amount") >= 200})
    .where("order_id <= 4")
    .select("order_id", "order_date", "order_hour", "is_large")
    .orderBy("order_id")
)
result.show()

## Exercise 3: Normalize messy column names

**Difficulty:** Medium · about 10 min · **Practises:** [Renaming: `withColumnRenamed`, `withColumnsRenamed`, `toDF`, `alias`](pyspark-reference-02-dataframe-api.ipynb#renaming-withcolumnrenamed-withcolumnsrenamed-todf-alias)

*Vendor CSVs arrive with spaces, capitals and punctuation in headers.*

Rename every column of `vendor` to lower snake case (lowercase, runs of non-alphanumeric characters become one `_`, no leading or trailing `_`). Return one row per column with `position` (1-based) and `name`.

In [ ]:
vendor = spark.createDataFrame([(1, "x", 2.0, "y")], "`Order ID` int, `Customer-Name ` string, `Unit Price (EUR)` double, `ship.to` string")
print(vendor.columns)

**Expected output**

| position | name |
| --- | --- |
| 1 | order_id |
| 2 | customer_name |
| 3 | unit_price_eur |
| 4 | ship_to |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
import re as _re
clean = vendor.toDF(*[_re.sub(r"[^0-9a-z]+", "_", c.lower()).strip("_") for c in vendor.columns])
result = spark.createDataFrame(list(enumerate(clean.columns, 1)), "position int, name string")
result.show()

## Exercise 4: Select columns with awkward names

**Difficulty:** Easy · about 5 min · **Practises:** [Referring to columns](pyspark-reference-02-dataframe-api.ipynb#referring-to-columns)

*Backticks are needed whenever a name contains a dot or space.*

From `odd`, return `order_id` (from the column `order.id`), `price` (from `unit price`) and `field` (from the struct field `s.field`).

In [ ]:
odd = spark.createDataFrame([(1, 9, (3,)), (2, 4, (5,))], "`order.id` int, `unit price` int, s struct<field:int>")
odd.show()

**Expected output**

| order_id | price | field |
| --- | --- | --- |
| 1 | 9 | 3 |
| 2 | 4 | 5 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = odd.select(F.col("`order.id`").alias("order_id"), F.col("`unit price`").alias("price"), F.col("s.field").alias("field")).orderBy("order_id")
result.show()

## Exercise 5: Keep rows where a nullable column is not a value

**Difficulty:** Easy · about 5 min · **Practises:** [`where(condition)` / `filter(condition)`](pyspark-reference-02-dataframe-api.ipynb#wherecondition--filtercondition)

*A classic silent bug: `!=` drops null rows.*

Return `customer_id` and `region` of customers whose region is **not** `North`, including customers with no region, ordered by `customer_id`.

In [ ]:
customers.select("customer_id", "region").show()

**Expected output**

| customer_id | region |
| --- | --- |
| 3 | South |
| 4 | null |
| 5 | West |
| 6 | West |
| 7 | East |
| 8 | East |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = (
    customers.where((F.col("region") != "North") | F.col("region").isNull())
    .select("customer_id", "region").orderBy("customer_id")
)
result.show()

## Exercise 6: Remove exact duplicate events

**Difficulty:** Easy · about 5 min · **Practises:** [`distinct()` and `dropDuplicates(subset=None)`](pyspark-reference-02-dataframe-api.ipynb#distinct-and-dropduplicatessubsetnone)

*SDK retries create exact duplicates that inflate funnel counts.*

Return `user_id`, `event` and `events` (count of distinct events per user and event type) for user 5, ordered by `event`.

In [ ]:
events.where("user_id = 5").show()

**Expected output**

| user_id | event | events |
| --- | --- | --- |
| 5 | purchase | 1 |
| 5 | view | 2 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = events.distinct().where("user_id = 5").groupBy("user_id", "event").agg(F.count("*").alias("events")).orderBy("event")
result.show()

## Exercise 7: Unique values in a column

**Difficulty:** Easy · about 5 min · **Practises:** [`distinct()` and `dropDuplicates(subset=None)`](pyspark-reference-02-dataframe-api.ipynb#distinct-and-dropduplicatessubsetnone)

*Profiling a column's domain is the first check on a new source.*

Return the distinct `status` values of `orders` and `orders` (how many orders have each), ordered by `status`.

In [ ]:
orders.select("status").show()

**Expected output**

| status | orders |
| --- | --- |
| paid | 4 |
| placed | 2 |
| returned | 2 |
| shipped | 4 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = orders.groupBy("status").agg(F.count("*").alias("orders")).orderBy("status")
result.show()

## Exercise 8: Merge feeds with different column orders

**Difficulty:** Medium · about 10 min · **Practises:** [`union`, `unionAll` and `unionByName`](pyspark-reference-02-dataframe-api.ipynb#union-unionall-and-unionbyname)

*Two systems export the same entity with different layouts; a positional union corrupts data.*

Combine `web`, `store` and `app` into one DataFrame matched by column name, with `channel` null where a feed does not have it. Return `order_id`, `status`, `channel`, ordered by `order_id`.

In [ ]:
web = spark.createDataFrame([(1, "paid", "web")], "order_id int, status string, channel string")
store = spark.createDataFrame([("shipped", 2)], "status string, order_id int")
app = spark.createDataFrame([(3, "app", "placed")], "order_id int, channel string, status string")
web.show(); store.show(); app.show()

**Expected output**

| order_id | status | channel |
| --- | --- | --- |
| 1 | paid | web |
| 2 | shipped | null |
| 3 | placed | app |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = web.unionByName(store, allowMissingColumns=True).unionByName(app).select("order_id", "status", "channel").orderBy("order_id")
result.show()

## Exercise 9: Compare two snapshots including duplicates

**Difficulty:** Medium · about 10 min · **Practises:** [`intersect`, `subtract` and `exceptAll`](pyspark-reference-02-dataframe-api.ipynb#intersect-subtract-and-exceptall)

*Regression tests compare expected and actual tables; duplicates must count.*

Return `check` and `rows` for: `expected.exceptAll(actual)`, `actual.exceptAll(expected)`, `expected.subtract(actual)` (row counts), in that order.

In [ ]:
expected = spark.createDataFrame([(1, "a"), (1, "a"), (2, "b"), (3, "c")], "id int, v string")
actual = spark.createDataFrame([(1, "a"), (2, "b"), (3, "c"), (4, "d")], "id int, v string")
expected.show(); actual.show()

**Expected output**

| check | rows |
| --- | --- |
| expected_exceptAll_actual | 1 |
| actual_exceptAll_expected | 1 |
| expected_subtract_actual | 0 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = spark.createDataFrame([
    ("expected_exceptAll_actual", expected.exceptAll(actual).count()),
    ("actual_exceptAll_expected", actual.exceptAll(expected).count()),
    ("expected_subtract_actual", expected.subtract(actual).count()),
], "check string, rows int")
result.show()

## Exercise 10: Customers present in both lists

**Difficulty:** Easy · about 5 min · **Practises:** [`intersect`, `subtract` and `exceptAll`](pyspark-reference-02-dataframe-api.ipynb#intersect-subtract-and-exceptall)

*Audience overlap between two campaigns.*

Return `customer_id` of customers in both `campaign_a` and `campaign_b`, without duplicates, ordered.

In [ ]:
campaign_a = spark.createDataFrame([(1,), (2,), (2,), (5,)], "customer_id int")
campaign_b = spark.createDataFrame([(2,), (5,), (7,)], "customer_id int")

**Expected output**

| customer_id |
| --- |
| 2 |
| 5 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = campaign_a.intersect(campaign_b).orderBy("customer_id")
result.show()

## Exercise 11: Paginate a sorted result

**Difficulty:** Medium · about 10 min · **Practises:** [`limit(n)`, `offset(n)` and `tail(n)`](pyspark-reference-02-dataframe-api.ipynb#limitn-offsetn-and-tailn)

*API backends and exports page through results with a stable order.*

With pages of 4 orders sorted by `amount` descending then `order_id`, return page 2 (the 5th to 8th orders) as `order_id`, `amount`.

In [ ]:
orders.select("order_id", "amount").orderBy(F.desc("amount")).show()

**Expected output**

| order_id | amount |
| --- | --- |
| 6 | 149.00 |
| 1 | 120.00 |
| 12 | 96.40 |
| 8 | 89.90 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = orders.orderBy(F.desc("amount"), "order_id").offset(4).limit(4).select("order_id", "amount")
result.show()

## Exercise 12: Sort with nulls last and a tie-breaker

**Difficulty:** Easy · about 5 min · **Practises:** [`orderBy(*cols, ascending=True)` / `sort(...)`](pyspark-reference-02-dataframe-api.ipynb#orderbycols-ascendingtrue--sort)

*Reports must not lead with unknowns, and must sort the same way every run.*

Return `customer_id`, `name` and `region` sorted by region descending with nulls last, then by `name`.

In [ ]:
customers.show()

**Expected output**

| customer_id | name | region |
| --- | --- | --- |
| 5 | Echo Supplies | West |
| 6 | Foxtrot Foods | West |
| 3 | Cyan Traders | South |
| 1 | Acme Corp | North |
| 2 | Bright Retail | North |
| 7 | Gamma Sports | East |
| 8 | Hotel Homeware | East |
| 4 | Delta Grocers | null |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = customers.select("customer_id", "name", "region").orderBy(F.col("region").desc_nulls_last(), "name")
result.show()

## Exercise 13: Build a pipeline with transform

**Difficulty:** Medium · about 10 min · **Practises:** [`DataFrame.transform(func, *args, **kwargs)`](pyspark-reference-02-dataframe-api.ipynb#dataframetransformfunc-args-kwargs)

*Small testable functions chained with transform is the standard shape of production PySpark.*

Write `drop_returns(df)`, `add_gross(df, rate)` and `keep_large(df, threshold)` and chain them with `transform` on `orders` (rate 0.2, threshold 200 on gross). Return `order_id` and `gross` (rounded to 2), ordered by `order_id`.

In [ ]:
orders.select("order_id", "status", "amount").show()

**Expected output**

| order_id | gross |
| --- | --- |
| 4 | 312.0 |
| 7 | 372.0 |
| 9 | 1416.0 |
| 11 | 252.0 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def drop_returns(df):
    return df.where(F.col("status") != "returned")

def add_gross(df, rate):
    return df.withColumn("gross", F.round(F.col("amount") * (1 + rate), 2))

def keep_large(df, threshold):
    return df.where(F.col("gross") >= threshold)

result = orders.transform(drop_returns).transform(add_gross, 0.2).transform(keep_large, 200).select("order_id", "gross").orderBy("order_id")
result.show()

## Exercise 14: Summary statistics for a column

**Difficulty:** Easy · about 5 min · **Practises:** [`describe(*cols)` and `summary(*statistics)`](pyspark-reference-02-dataframe-api.ipynb#describecols-and-summarystatistics)

*A quick profile of a numeric column before setting quality thresholds.*

Return the `count`, `min`, `max` and `count_distinct` rows of `summary` for `amount` and `customer_id`, as columns `summary`, `amount`, `customer_id`.

In [ ]:
orders.select("amount", "customer_id").show()

**Expected output**

| summary | amount | customer_id |
| --- | --- | --- |
| count | 12 | 12 |
| min | 30.00 | 1 |
| max | 1180.00 | 6 |
| count_distinct | 12 | 6 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = orders.select("amount", "customer_id").summary("count", "min", "max", "count_distinct")
result.show()

## Exercise 15: Create a nested DataFrame from Python data

**Difficulty:** Hard · about 15 min · **Practises:** [Schemas: `StructType`, `StructField` and DDL strings](pyspark-reference-02-dataframe-api.ipynb#schemas-structtype-structfield-and-ddl-strings)

*Test fixtures for nested JSON sources need explicit nested schemas.*

Build a DataFrame from `payload` with schema `order_id int, customer struct<id:int,name:string>, items array<struct<sku:string,qty:int>>`, then return `order_id`, `customer_name` and `units` (sum of `qty` across items, using `aggregate`), ordered by `order_id`.

In [ ]:
payload = [
    (1, (7, "Gamma Sports"), [("SKU-1", 2), ("SKU-2", 1)]),
    (2, (8, "Hotel Homeware"), [("SKU-3", 5)]),
]
print(payload)

**Expected output**

| order_id | customer_name | units |
| --- | --- | --- |
| 1 | Gamma Sports | 3 |
| 2 | Hotel Homeware | 5 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
nested = spark.createDataFrame(payload, "order_id int, customer struct<id:int,name:string>, items array<struct<sku:string,qty:int>>")
result = nested.select(
    "order_id",
    F.col("customer.name").alias("customer_name"),
    F.aggregate("items", F.lit(0), lambda total, item: total + item["qty"]).alias("units"),
).orderBy("order_id")
result.show()

## Exercise 16: Detect schema drift between two versions

**Difficulty:** Hard · about 15 min · **Practises:** [Schemas: `StructType`, `StructField` and DDL strings](pyspark-reference-02-dataframe-api.ipynb#schemas-structtype-structfield-and-ddl-strings)

*Pipelines should fail fast when a source adds, drops or retypes a column.*

Compare `v1` and `v2` by column name. Return `column`, `v1_type`, `v2_type` and `status` (`added`, `removed`, `type_changed`) for every column that differs, ordered by `column`.

In [ ]:
v1 = spark.createDataFrame([], "order_id int, amount decimal(12,2), status string, note string")
v2 = spark.createDataFrame([], "order_id bigint, amount decimal(12,2), status string, channel string")
print(v1.dtypes); print(v2.dtypes)

**Expected output**

| column | v1_type | v2_type | status |
| --- | --- | --- | --- |
| channel | null | string | added |
| note | string | null | removed |
| order_id | int | bigint | type_changed |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
old, new = dict(v1.dtypes), dict(v2.dtypes)
rows = []
for col in sorted(set(old) | set(new)):
    if col not in old:
        rows.append((col, None, new[col], "added"))
    elif col not in new:
        rows.append((col, old[col], None, "removed"))
    elif old[col] != new[col]:
        rows.append((col, old[col], new[col], "type_changed"))
result = spark.createDataFrame(rows, "column string, v1_type string, v2_type string, status string")
result.show()

**Next:** look anything up in [PySpark reference 2: DataFrame API](pyspark-reference-02-dataframe-api.ipynb), or test yourself across topics with the [PySpark coding challenges](pyspark-coding-challenges.ipynb).